# Relational Databases for Data Science. Домашнє завдання до Теми 2
## Проектування баз даних з використанням семантичних моделей

**Сценарій:** Customer Churn Prediction

Публічний churn dataset використовується лише як доменний референс. Для temporal-частини використовується контрольований набір історії, тому точний churn timestamp не припускається як відомий із зовнішнього dataset.

Мета роботи - спроєктувати PostgreSQL-схему, яка підтримує business data, offline та online feature store, training events і point-in-time correct формування training dataset.

## 1. ER-модель

Природні сутності:
- `customers` - клієнти;
- `service_plans` - тарифні плани;
- `subscriptions` - факт підписки клієнта на тариф;
- `payments` - платежі за підпискою.

Зв'язок `customers <-> service_plans` є природним N:M у часі та реалізований через асоціативну таблицю `subscriptions`.

Окремо виділено:
- `customer_features_offline` - історія ознак для training;
- `customer_features_online` - поточний денормалізований snapshot для inference;
- `training_events` - моменти прогнозу і label.

### Текстова ER-схема

```text
CUSTOMERS 1 ----- N SUBSCRIPTIONS N ----- 1 SERVICE_PLANS
                      |
                      | 1
                      |
                      N
                   PAYMENTS

CUSTOMERS 1 ----- N CUSTOMER_FEATURES_OFFLINE
CUSTOMERS 1 ----- 1 CUSTOMER_FEATURES_ONLINE
CUSTOMERS 1 ----- N TRAINING_EVENTS

OFFLINE: (customer_id, valid_from) -> feature values, valid_to
ONLINE:  customer_id -> current denormalized feature values
EVENT:   event_id, customer_id, prediction_ts, label_horizon_end, churn_in_30d
```

## 2. Технічне середовище

Використовується PostgreSQL 16 через `pgserver` у Google Colab.

 **Note:** Обовʼязково перевірте щоб Runtime type був версії 2026.04. В оcтанній версії pgserver відсутній.

In [2]:
!pip install -q pgserver sqlalchemy psycopg2-binary pandas

import pgserver
import pandas as pd
from sqlalchemy import create_engine

pg = pgserver.get_server('/tmp/pg_data')
engine = create_engine(pg.get_uri())

with engine.connect() as conn:
    version = conn.exec_driver_sql("SELECT version()").scalar()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


## 3. DDL у PostgreSQL

Схема містить PK, FK, NOT NULL, domain CHECK, temporal CHECK, індекси і захист від перекривних temporal-версій.

In [3]:
ddl_sql = r"""
SET TIME ZONE 'UTC';

DROP TABLE IF EXISTS
    hw2_customer_features_online,
    hw2_training_events,
    hw2_customer_features_offline,
    hw2_payments,
    hw2_subscriptions,
    hw2_service_plans,
    hw2_customers
CASCADE;
DROP FUNCTION IF EXISTS hw2_prevent_feature_overlap() CASCADE;

CREATE TABLE hw2_customers (
    customer_id BIGINT PRIMARY KEY,
    signup_date DATE NOT NULL,
    country TEXT NOT NULL,
    birth_year INTEGER CHECK (birth_year IS NULL OR birth_year BETWEEN 1900 AND 2100)
);

CREATE TABLE hw2_service_plans (
    plan_id BIGINT PRIMARY KEY,
    plan_name TEXT NOT NULL UNIQUE,
    monthly_fee NUMERIC(12, 2) NOT NULL CHECK (monthly_fee >= 0),
    contract_type TEXT NOT NULL CHECK (contract_type IN ('monthly', 'annual'))
);

CREATE TABLE hw2_subscriptions (
    subscription_id BIGINT PRIMARY KEY,
    customer_id BIGINT NOT NULL REFERENCES hw2_customers(customer_id),
    plan_id BIGINT NOT NULL REFERENCES hw2_service_plans(plan_id),
    start_date DATE NOT NULL,
    end_date DATE,
    status TEXT NOT NULL CHECK (status IN ('active', 'paused', 'cancelled')),
    autopay BOOLEAN NOT NULL,
    CHECK (end_date IS NULL OR end_date >= start_date)
);
CREATE INDEX hw2_idx_subscriptions_customer ON hw2_subscriptions (customer_id, start_date DESC);

CREATE TABLE hw2_payments (
    payment_id BIGINT PRIMARY KEY,
    subscription_id BIGINT NOT NULL REFERENCES hw2_subscriptions(subscription_id),
    payment_ts TIMESTAMPTZ NOT NULL,
    amount NUMERIC(12, 2) NOT NULL CHECK (amount >= 0),
    paid_late BOOLEAN NOT NULL
);
CREATE INDEX hw2_idx_payments_subscription_ts ON hw2_payments (subscription_id, payment_ts DESC);

CREATE TABLE hw2_customer_features_offline (
    customer_id BIGINT NOT NULL REFERENCES hw2_customers(customer_id),
    tenure_days INTEGER NOT NULL CHECK (tenure_days >= 0),
    payments_90d INTEGER NOT NULL CHECK (payments_90d >= 0),
    avg_payment_90d NUMERIC(12, 2) CHECK (avg_payment_90d IS NULL OR avg_payment_90d >= 0),
    late_payments_90d INTEGER NOT NULL CHECK (late_payments_90d >= 0),
    active_subscription_count INTEGER NOT NULL CHECK (active_subscription_count >= 0),
    has_autopay BOOLEAN NOT NULL,
    valid_from TIMESTAMPTZ NOT NULL,
    valid_to TIMESTAMPTZ,
    PRIMARY KEY (customer_id, valid_from),
    CHECK (valid_to IS NULL OR valid_to > valid_from)
);
CREATE INDEX hw2_idx_customer_features_asof ON hw2_customer_features_offline (customer_id, valid_from DESC);

CREATE OR REPLACE FUNCTION hw2_prevent_feature_overlap()
RETURNS trigger LANGUAGE plpgsql AS $$
BEGIN
    IF EXISTS (
        SELECT 1
        FROM hw2_customer_features_offline AS f
        WHERE f.customer_id = NEW.customer_id
          AND tstzrange(f.valid_from, COALESCE(f.valid_to, 'infinity'::timestamptz), '[)')
              &&
              tstzrange(NEW.valid_from, COALESCE(NEW.valid_to, 'infinity'::timestamptz), '[)')
          AND NOT (f.customer_id = NEW.customer_id AND f.valid_from = NEW.valid_from)
    ) THEN
        RAISE EXCEPTION 'Overlapping feature interval for customer_id=%', NEW.customer_id;
    END IF;
    RETURN NEW;
END;
$$;

CREATE TRIGGER hw2_trg_prevent_feature_overlap
BEFORE INSERT OR UPDATE ON hw2_customer_features_offline
FOR EACH ROW EXECUTE FUNCTION hw2_prevent_feature_overlap();

CREATE TABLE hw2_training_events (
    event_id BIGINT GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    customer_id BIGINT NOT NULL REFERENCES hw2_customers(customer_id),
    prediction_ts TIMESTAMPTZ NOT NULL,
    label_horizon_end TIMESTAMPTZ NOT NULL,
    churn_in_30d BOOLEAN NOT NULL,
    CHECK (label_horizon_end > prediction_ts)
);
CREATE INDEX hw2_idx_training_events_customer_ts ON hw2_training_events (customer_id, prediction_ts);

CREATE TABLE hw2_customer_features_online (
    customer_id BIGINT PRIMARY KEY REFERENCES hw2_customers(customer_id),
    tenure_days INTEGER NOT NULL CHECK (tenure_days >= 0),
    payments_90d INTEGER NOT NULL CHECK (payments_90d >= 0),
    avg_payment_90d NUMERIC(12, 2) CHECK (avg_payment_90d IS NULL OR avg_payment_90d >= 0),
    late_payments_90d INTEGER NOT NULL CHECK (late_payments_90d >= 0),
    active_subscription_count INTEGER NOT NULL CHECK (active_subscription_count >= 0),
    has_autopay BOOLEAN NOT NULL,
    feature_ts TIMESTAMPTZ NOT NULL,
    updated_at TIMESTAMPTZ NOT NULL DEFAULT CURRENT_TIMESTAMP
);
"""

raw_conn = engine.raw_connection()
try:
    cur = raw_conn.cursor()
    cur.execute(ddl_sql)
    raw_conn.commit()
finally:
    raw_conn.close()

print("DDL executed successfully.")

DDL executed successfully.


## 4. Контрольовані business data

Записи потрібні для демонстрації структури домену. Temporal feature history створюється окремо, без припущення про реальний churn timestamp із публічного dataset.

In [4]:
business_data_sql = r"""
INSERT INTO hw2_customers (customer_id, signup_date, country, birth_year) VALUES
(1, DATE '2024-01-10', 'UA', 1988),
(2, DATE '2024-02-20', 'PL', 1992),
(3, DATE '2024-03-15', 'UA', 1979),
(42, DATE '2024-04-01', 'DE', 1985);

INSERT INTO hw2_service_plans (plan_id, plan_name, monthly_fee, contract_type) VALUES
(10, 'Basic', 19.99, 'monthly'),
(20, 'Plus', 39.99, 'monthly'),
(30, 'Premium', 59.99, 'annual');

INSERT INTO hw2_subscriptions (subscription_id, customer_id, plan_id, start_date, end_date, status, autopay) VALUES
(1001, 1, 10, DATE '2024-01-10', DATE '2024-08-31', 'cancelled', FALSE),
(1002, 1, 20, DATE '2024-09-01', NULL, 'active', TRUE),
(1003, 2, 20, DATE '2024-02-20', NULL, 'active', TRUE),
(1004, 3, 10, DATE '2024-03-15', NULL, 'active', FALSE),
(1005, 42, 30, DATE '2024-04-01', NULL, 'active', TRUE);

INSERT INTO hw2_payments (payment_id, subscription_id, payment_ts, amount, paid_late) VALUES
(5001, 1002, TIMESTAMPTZ '2024-11-05 08:00:00+00', 39.99, FALSE),
(5002, 1002, TIMESTAMPTZ '2024-12-05 08:00:00+00', 39.99, FALSE),
(5003, 1003, TIMESTAMPTZ '2024-11-07 08:00:00+00', 39.99, TRUE),
(5004, 1003, TIMESTAMPTZ '2024-12-07 08:00:00+00', 39.99, FALSE),
(5005, 1004, TIMESTAMPTZ '2024-11-10 08:00:00+00', 19.99, TRUE),
(5006, 1004, TIMESTAMPTZ '2024-12-10 08:00:00+00', 19.99, TRUE),
(5007, 1005, TIMESTAMPTZ '2024-11-12 08:00:00+00', 59.99, FALSE),
(5008, 1005, TIMESTAMPTZ '2024-12-12 08:00:00+00', 59.99, FALSE);
"""
raw_conn = engine.raw_connection()
try:
    cur = raw_conn.cursor(); cur.execute(business_data_sql); raw_conn.commit()
finally:
    raw_conn.close()

display(pd.read_sql("""
SELECT c.customer_id, c.country, s.subscription_id, p.plan_name, s.status, s.autopay
FROM hw2_customers AS c
LEFT JOIN hw2_subscriptions AS s ON s.customer_id = c.customer_id
LEFT JOIN hw2_service_plans AS p ON p.plan_id = s.plan_id
ORDER BY c.customer_id, s.subscription_id
""", engine))

,customer_id,country,subscription_id,plan_name,status,autopay
0,1,UA,1001,Basic,cancelled,False
1,1,UA,1002,Plus,active,True
2,2,PL,1003,Plus,active,True
3,3,UA,1004,Basic,active,False
4,42,DE,1005,Premium,active,True


## 5. Історична offline feature-таблиця

Використовуємо 12 temporal-записів для 4 customers. Для кожного customer є три послідовні версії feature. Інтервал валідності - `[valid_from, valid_to)`.

In [5]:
feature_history_sql = r"""
INSERT INTO hw2_customer_features_offline
(customer_id, tenure_days, payments_90d, avg_payment_90d, late_payments_90d, active_subscription_count, has_autopay, valid_from, valid_to)
VALUES
(1,265,2,29.99,0,1,TRUE, TIMESTAMPTZ '2024-10-01 00:00:00+00', TIMESTAMPTZ '2024-11-15 00:00:00+00'),
(1,310,3,36.66,0,1,TRUE, TIMESTAMPTZ '2024-11-15 00:00:00+00', TIMESTAMPTZ '2024-12-20 00:00:00+00'),
(1,345,3,39.99,0,1,TRUE, TIMESTAMPTZ '2024-12-20 00:00:00+00', NULL),
(2,224,3,39.99,2,1,TRUE, TIMESTAMPTZ '2024-10-01 00:00:00+00', TIMESTAMPTZ '2024-11-10 00:00:00+00'),
(2,264,3,39.99,1,1,TRUE, TIMESTAMPTZ '2024-11-10 00:00:00+00', TIMESTAMPTZ '2024-12-18 00:00:00+00'),
(2,302,3,39.99,0,1,TRUE, TIMESTAMPTZ '2024-12-18 00:00:00+00', NULL),
(3,200,2,19.99,2,1,FALSE, TIMESTAMPTZ '2024-10-01 00:00:00+00', TIMESTAMPTZ '2024-11-20 00:00:00+00'),
(3,250,2,19.99,2,1,FALSE, TIMESTAMPTZ '2024-11-20 00:00:00+00', TIMESTAMPTZ '2024-12-25 00:00:00+00'),
(3,285,1,19.99,1,1,FALSE, TIMESTAMPTZ '2024-12-25 00:00:00+00', NULL),
(42,183,2,59.99,0,1,TRUE, TIMESTAMPTZ '2024-10-01 00:00:00+00', TIMESTAMPTZ '2024-11-01 00:00:00+00'),
(42,214,3,59.99,0,1,TRUE, TIMESTAMPTZ '2024-11-01 00:00:00+00', TIMESTAMPTZ '2024-12-15 00:00:00+00'),
(42,258,3,59.99,0,1,TRUE, TIMESTAMPTZ '2024-12-15 00:00:00+00', NULL);
"""
raw_conn = engine.raw_connection()
try:
    cur=raw_conn.cursor(); cur.execute(feature_history_sql); raw_conn.commit()
finally:
    raw_conn.close()

history = pd.read_sql("SELECT * FROM hw2_customer_features_offline ORDER BY customer_id, valid_from", engine)
print("Historical rows:", len(history))
display(history)
assert len(history) >= 10
assert history.groupby("customer_id").size().max() >= 3

Historical rows: 12


,customer_id,tenure_days,payments_90d,avg_payment_90d,late_payments_90d,active_subscription_count,has_autopay,valid_from,valid_to
0,1,265,2,29.99,0,1,True,2024-10-01 00:00:00+00:00,2024-11-15 00:00:00+00:00
1,1,310,3,36.66,0,1,True,2024-11-15 00:00:00+00:00,2024-12-20 00:00:00+00:00
2,1,345,3,39.99,0,1,True,2024-12-20 00:00:00+00:00,NaT
3,2,224,3,39.99,2,1,True,2024-10-01 00:00:00+00:00,2024-11-10 00:00:00+00:00
4,2,264,3,39.99,1,1,True,2024-11-10 00:00:00+00:00,2024-12-18 00:00:00+00:00
5,2,302,3,39.99,0,1,True,2024-12-18 00:00:00+00:00,NaT
6,3,200,2,19.99,2,1,False,2024-10-01 00:00:00+00:00,2024-11-20 00:00:00+00:00
7,3,250,2,19.99,2,1,False,2024-11-20 00:00:00+00:00,2024-12-25 00:00:00+00:00
8,3,285,1,19.99,1,1,False,2024-12-25 00:00:00+00:00,NaT
9,42,183,2,59.99,0,1,True,2024-10-01 00:00:00+00:00,2024-11-01 00:00:00+00:00


## 6. Training events

`prediction_ts` - момент, на який формується training example. `label_horizon_end` - кінець 30-денного горизонту для label `churn_in_30d`.

In [6]:
events_sql = r"""
INSERT INTO hw2_training_events (customer_id, prediction_ts, label_horizon_end, churn_in_30d) VALUES
(1, TIMESTAMPTZ '2024-11-01 10:00:00+00', TIMESTAMPTZ '2024-12-01 10:00:00+00', FALSE),
(1, TIMESTAMPTZ '2024-12-22 10:00:00+00', TIMESTAMPTZ '2025-01-21 10:00:00+00', FALSE),
(2, TIMESTAMPTZ '2024-11-15 10:00:00+00', TIMESTAMPTZ '2024-12-15 10:00:00+00', FALSE),
(2, TIMESTAMPTZ '2024-12-20 10:00:00+00', TIMESTAMPTZ '2025-01-19 10:00:00+00', FALSE),
(3, TIMESTAMPTZ '2024-12-10 10:00:00+00', TIMESTAMPTZ '2025-01-09 10:00:00+00', TRUE),
(42, TIMESTAMPTZ '2024-12-10 10:00:00+00', TIMESTAMPTZ '2025-01-09 10:00:00+00', FALSE);
"""
raw_conn = engine.raw_connection()
try:
    cur=raw_conn.cursor(); cur.execute(events_sql); raw_conn.commit()
finally:
    raw_conn.close()

display(pd.read_sql("SELECT * FROM hw2_training_events ORDER BY event_id", engine))

,event_id,customer_id,prediction_ts,label_horizon_end,churn_in_30d
0,1,1,2024-11-01 10:00:00+00:00,2024-12-01 10:00:00+00:00,False
1,2,1,2024-12-22 10:00:00+00:00,2025-01-21 10:00:00+00:00,False
2,3,2,2024-11-15 10:00:00+00:00,2024-12-15 10:00:00+00:00,False
3,4,2,2024-12-20 10:00:00+00:00,2025-01-19 10:00:00+00:00,False
4,5,3,2024-12-10 10:00:00+00:00,2025-01-09 10:00:00+00:00,True
5,6,42,2024-12-10 10:00:00+00:00,2025-01-09 10:00:00+00:00,False


## 7. Online feature store

Online-рівень навмисно денормалізований: один рядок містить готовий набір поточних features для одного `customer_id`. Оновлення виконується з offline source через вибір останньої версії.

In [7]:
online_sql = r"""
INSERT INTO hw2_customer_features_online
(customer_id, tenure_days, payments_90d, avg_payment_90d, late_payments_90d, active_subscription_count, has_autopay, feature_ts)
SELECT DISTINCT ON (customer_id)
    customer_id, tenure_days, payments_90d, avg_payment_90d, late_payments_90d,
    active_subscription_count, has_autopay, valid_from
FROM hw2_customer_features_offline
ORDER BY customer_id, valid_from DESC;
"""
raw_conn = engine.raw_connection()
try:
    cur=raw_conn.cursor(); cur.execute(online_sql); raw_conn.commit()
finally:
    raw_conn.close()

display(pd.read_sql("SELECT * FROM hw2_customer_features_online ORDER BY customer_id", engine))

,customer_id,tenure_days,payments_90d,avg_payment_90d,late_payments_90d,active_subscription_count,has_autopay,feature_ts,updated_at
0,1,345,3,39.99,0,1,True,2024-12-20 00:00:00+00:00,2026-10-11 17:26:53.494191+00:00
1,2,302,3,39.99,0,1,True,2024-12-18 00:00:00+00:00,2026-10-11 17:26:53.494191+00:00
2,3,285,1,19.99,1,1,False,2024-12-25 00:00:00+00:00,2026-10-11 17:26:53.494191+00:00
3,42,258,3,59.99,0,1,True,2024-12-15 00:00:00+00:00,2026-10-11 17:26:53.494191+00:00


## 8. Нормалізація і денормалізація

Основні business tables спроєктовані у 3НФ. `hw2_customers` містить лише факти про клієнта, `hw2_service_plans` - лише факти про тариф, `hw2_subscriptions` - факти про конкретний зв'язок customer-plan у певний період, а `hw2_payments` - факти про окремі платежі. Наприклад, `plan_name` і `monthly_fee` не дублюються у `subscriptions`, бо функціонально залежать від `plan_id`, а не від `subscription_id`. Аналогічно payment attributes залежать від `payment_id`, а customer attributes не дублюються в payments.

Таблиця `hw2_customer_features_offline` зберігає історичні snapshots ознак. Її ключ `(customer_id, valid_from)` визначає конкретну temporal-версію features. Ця таблиця орієнтована на ML access pattern, а не на OLTP.

`hw2_customer_features_online` денормалізована свідомо. Для inference головний сценарій читання - отримати всі потрібні features одним lookup за `customer_id` з низькою latency. Це дозволяє уникнути JOIN між customers, subscriptions і payments під час кожного prediction request.

Ризик денормалізації - неузгодженість між offline та online values. Тому online table не є незалежним source of truth: вона матеріалізується з останніх offline versions. У production цей процес виконувався б materialization pipeline з контролем freshness, а зміни schema і feature semantics були б версіонованими.

## 9. Point-in-time correctness

Коректна temporal умова:

```sql
valid_from <= prediction_ts
AND (valid_to > prediction_ts OR valid_to IS NULL)
```

Вона реалізує напіввідкритий інтервал `[valid_from, valid_to)` і не дозволяє training example бачити майбутню feature version.

### AS-OF SELECT 1 - customer 42 до зміни 2024-12-15

In [8]:
asof_1 = pd.read_sql("""
SELECT customer_id, tenure_days, payments_90d, avg_payment_90d, late_payments_90d, valid_from, valid_to
FROM hw2_customer_features_offline
WHERE customer_id = 42
  AND valid_from <= TIMESTAMPTZ '2024-12-10 10:00:00+00'
  AND (valid_to > TIMESTAMPTZ '2024-12-10 10:00:00+00' OR valid_to IS NULL);
""", engine)
display(asof_1)
assert len(asof_1) == 1
assert int(asof_1.iloc[0]["tenure_days"]) == 214

,customer_id,tenure_days,payments_90d,avg_payment_90d,late_payments_90d,valid_from,valid_to
0,42,214,3,59.99,0,2024-11-01 00:00:00+00:00,2024-12-15 00:00:00+00:00


### AS-OF SELECT 2 - customer 42 після зміни 2024-12-15

In [9]:
asof_2 = pd.read_sql("""
SELECT customer_id, tenure_days, payments_90d, avg_payment_90d, late_payments_90d, valid_from, valid_to
FROM hw2_customer_features_offline
WHERE customer_id = 42
  AND valid_from <= TIMESTAMPTZ '2024-12-20 10:00:00+00'
  AND (valid_to > TIMESTAMPTZ '2024-12-20 10:00:00+00' OR valid_to IS NULL);
""", engine)
display(asof_2)
assert len(asof_2) == 1
assert int(asof_2.iloc[0]["tenure_days"]) == 258

,customer_id,tenure_days,payments_90d,avg_payment_90d,late_payments_90d,valid_from,valid_to
0,42,258,3,59.99,0,2024-12-15 00:00:00+00:00,None


### AS-OF SELECT 3 - training-data JOIN за prediction_ts

In [10]:
training_data = pd.read_sql("""
SELECT
    e.event_id, e.customer_id, e.prediction_ts, e.label_horizon_end, e.churn_in_30d,
    f.tenure_days, f.payments_90d, f.avg_payment_90d, f.late_payments_90d,
    f.active_subscription_count, f.has_autopay,
    f.valid_from AS feature_valid_from, f.valid_to AS feature_valid_to
FROM hw2_training_events AS e
LEFT JOIN hw2_customer_features_offline AS f
    ON f.customer_id = e.customer_id
   AND f.valid_from <= e.prediction_ts
   AND (f.valid_to > e.prediction_ts OR f.valid_to IS NULL)
ORDER BY e.event_id;
""", engine)
display(training_data)
assert len(training_data) == 6
assert training_data["tenure_days"].notna().all()
assert (training_data["feature_valid_from"] <= training_data["prediction_ts"]).all()

,event_id,customer_id,prediction_ts,label_horizon_end,churn_in_30d,tenure_days,payments_90d,avg_payment_90d,late_payments_90d,active_subscription_count,has_autopay,feature_valid_from,feature_valid_to
0,1,1,2024-11-01 10:00:00+00:00,2024-12-01 10:00:00+00:00,False,265,2,29.99,0,1,True,2024-10-01 00:00:00+00:00,2024-11-15 00:00:00+00:00
1,2,1,2024-12-22 10:00:00+00:00,2025-01-21 10:00:00+00:00,False,345,3,39.99,0,1,True,2024-12-20 00:00:00+00:00,NaT
2,3,2,2024-11-15 10:00:00+00:00,2024-12-15 10:00:00+00:00,False,264,3,39.99,1,1,True,2024-11-10 00:00:00+00:00,2024-12-18 00:00:00+00:00
3,4,2,2024-12-20 10:00:00+00:00,2025-01-19 10:00:00+00:00,False,302,3,39.99,0,1,True,2024-12-18 00:00:00+00:00,NaT
4,5,3,2024-12-10 10:00:00+00:00,2025-01-09 10:00:00+00:00,True,250,2,19.99,2,1,False,2024-11-20 00:00:00+00:00,2024-12-25 00:00:00+00:00
5,6,42,2024-12-10 10:00:00+00:00,2025-01-09 10:00:00+00:00,False,214,3,59.99,0,1,True,2024-11-01 00:00:00+00:00,2024-12-15 00:00:00+00:00


### Додатковий AS-OF SELECT - customer 3

In [11]:
asof_3 = pd.read_sql("""
SELECT customer_id, tenure_days, late_payments_90d, valid_from, valid_to
FROM hw2_customer_features_offline
WHERE customer_id = 3
  AND valid_from <= TIMESTAMPTZ '2024-12-10 10:00:00+00'
  AND (valid_to > TIMESTAMPTZ '2024-12-10 10:00:00+00' OR valid_to IS NULL);
""", engine)
display(asof_3)
assert len(asof_3) == 1
assert int(asof_3.iloc[0]["tenure_days"]) == 250

,customer_id,tenure_days,late_payments_90d,valid_from,valid_to
0,3,250,2,2024-11-20 00:00:00+00:00,2024-12-25 00:00:00+00:00


### Перевірка перекривних temporal-інтервалів

Запит має повернути 0 rows.

In [12]:
overlaps = pd.read_sql("""
SELECT a.customer_id, a.valid_from AS a_from, a.valid_to AS a_to, b.valid_from AS b_from, b.valid_to AS b_to
FROM hw2_customer_features_offline AS a
JOIN hw2_customer_features_offline AS b
  ON a.customer_id = b.customer_id
 AND a.valid_from < COALESCE(b.valid_to, 'infinity'::timestamptz)
 AND b.valid_from < COALESCE(a.valid_to, 'infinity'::timestamptz)
 AND a.valid_from < b.valid_from;
""", engine)
display(overlaps)
assert overlaps.empty
print("OK: no overlapping temporal versions.")

,customer_id,a_from,a_to,b_from,b_to


OK: no overlapping temporal versions.


### Чому JOIN лише за customer_id неправильний

Простий JOIN між `training_events` і `customer_features_offline` тільки за `customer_id` приєднає до кожної training event усі історичні версії features цього клієнта. Це створить дублікати і може приєднати версії, які з'явилися після `prediction_ts`.

Наприклад, event customer 42 на `2024-12-10` отримав би також feature version, яка почалася `2024-12-15`. Для історичної події це майбутня інформація, тобто data leakage.

In [13]:
bad_join = pd.read_sql("""
SELECT e.event_id, e.customer_id, e.prediction_ts, f.valid_from, f.valid_to
FROM hw2_training_events AS e
JOIN hw2_customer_features_offline AS f ON f.customer_id = e.customer_id
ORDER BY e.event_id, f.valid_from;
""", engine)
print("Training events:", len(pd.read_sql("SELECT * FROM hw2_training_events", engine)))
print("Rows after WRONG join:", len(bad_join))
display(bad_join.head(12))

Training events: 6
Rows after WRONG join: 18


,event_id,customer_id,prediction_ts,valid_from,valid_to
0,1,1,2024-11-01 10:00:00+00:00,2024-10-01 00:00:00+00:00,2024-11-15 00:00:00+00:00
1,1,1,2024-11-01 10:00:00+00:00,2024-11-15 00:00:00+00:00,2024-12-20 00:00:00+00:00
2,1,1,2024-11-01 10:00:00+00:00,2024-12-20 00:00:00+00:00,NaT
3,2,1,2024-12-22 10:00:00+00:00,2024-10-01 00:00:00+00:00,2024-11-15 00:00:00+00:00
4,2,1,2024-12-22 10:00:00+00:00,2024-11-15 00:00:00+00:00,2024-12-20 00:00:00+00:00
5,2,1,2024-12-22 10:00:00+00:00,2024-12-20 00:00:00+00:00,NaT
6,3,2,2024-11-15 10:00:00+00:00,2024-10-01 00:00:00+00:00,2024-11-10 00:00:00+00:00
7,3,2,2024-11-15 10:00:00+00:00,2024-11-10 00:00:00+00:00,2024-12-18 00:00:00+00:00
8,3,2,2024-11-15 10:00:00+00:00,2024-12-18 00:00:00+00:00,NaT
9,4,2,2024-12-20 10:00:00+00:00,2024-10-01 00:00:00+00:00,2024-11-10 00:00:00+00:00


## 10. Автоматична перевірка ключових критеріїв

In [14]:
checks = {
    ">= 10 historical feature rows": pd.read_sql("SELECT COUNT(*) AS n FROM hw2_customer_features_offline", engine).iloc[0]["n"] >= 10,
    ">= 3 entity_id": pd.read_sql("SELECT COUNT(DISTINCT customer_id) AS n FROM hw2_customer_features_offline", engine).iloc[0]["n"] >= 3,
    "at least one entity has >= 3 versions": pd.read_sql("""
        SELECT MAX(n) AS max_versions
        FROM (SELECT customer_id, COUNT(*) AS n FROM hw2_customer_features_offline GROUP BY customer_id) AS q
    """, engine).iloc[0]["max_versions"] >= 3,
    "overlap check returns zero rows": overlaps.empty,
    "training join returns one row per event": len(training_data) == len(pd.read_sql("SELECT * FROM hw2_training_events", engine)),
}
display(pd.DataFrame([{"check": k, "passed": bool(v)} for k, v in checks.items()]))
assert all(checks.values())

,check,passed
0,>= 10 historical feature rows,True
1,>= 3 entity_id,True
2,at least one entity has >= 3 versions,True
3,overlap check returns zero rows,True
4,training join returns one row per event,True


## 11. Reflection

Найскладнішою частиною роботи була не побудова самих таблиць, а визначення часової семантики features. Для customer churn недостатньо мати поточний snapshot клієнта: training example повинен бачити лише інформацію, яка існувала на `prediction_ts`. Тому offline feature table зберігає кілька версій для одного customer, а інтервали валідності трактуються як `[valid_from, valid_to)`. Це дозволяє однозначно визначити правильну версію навіть на межі двох інтервалів.

Business tables я залишив нормалізованими. Customers, plans, subscriptions і payments описують різні факти, тому їх змішування створило б дублювання та аномалії оновлення. Online feature table, навпаки, денормалізована свідомо. Під час inference головний access pattern - швидкий lookup одного customer за `entity_id`, тому готовий wide row практичніший за кілька JOIN у критичному request path. Компроміс полягає в тому, що online data дублює обчислені values з offline layer і може застаріти. Це потрібно контролювати materialization pipeline і freshness monitoring.

Якщо кількість користувачів і QPS значно зростуть, online-рівень я б виніс у low-latency key-value storage або distributed cache, зберігаючи денормалізований запис за `customer_id`. Додатково потрібні horizontal scaling, freshness control і monitoring lag.

За шість місяців схема може змінитися через нові features, перейменування, зміну типів або semantic changes. Найнебезпечніша ситуація - змінити зміст feature без зміни назви, бо це створює training-serving skew. Тому нову семантику краще версіонувати, наприклад `avg_payment_90d_v2`, а старі колонки не видаляти, поки їх використовує хоча б одна production model.

## 12. Підсумковий checklist

- ER-модель містить щонайменше три природні сутності.
- PK, FK, кардинальності та optionality описані.
- Offline і online feature store розділені.
- DDL використовує префікс `hw2_`.
- Є NOT NULL, domain CHECK, temporal CHECK та індекси.
- Є 12 historical feature rows для 4 customers.
- Для customer 42 є щонайменше 3 feature versions.
- Є щонайменше три AS-OF SELECT приклади.
- Training-data JOIN виконується за `prediction_ts`.
- Перевірка overlaps повертає 0 rows.
- Online layer матеріалізується з offline source.
- Є аналіз нормалізації/денормалізації та Reflection.